# DPO training: Socratic Tutor v6 -> v6+DPO

Runs on Colab (CUDA + bitsandbytes + peft + trl -- MLX has no DPO trainer). Takes the
already-fused `adapters/v6` SFT checkpoint (pushed to a private HF Hub repo from the local
side via `scripts/prepare_dpo_colab.py --push-fused`) and trains a light preference-tuning
pass on the mined chosen/rejected pairs (`data/dpo/dpo_pairs_rendered.jsonl`), then pushes
the merged result back to HF Hub for local `mlx_lm convert` + `eval_harness.py` evaluation.

**Before running:**
1. Set `HF_USERNAME` in the config cell below.
2. Upload `dpo_pairs_rendered.jsonl` via the Colab file browser (left sidebar), or mount
   Drive and point `PAIRS_PATH` at it there.
3. Set an `HF_TOKEN` secret (key icon, left sidebar) with `write` scope and notebook access
   enabled -- this notebook reads it via `google.colab.userdata`, never pasted inline.

**If the fail-fast load gate (a few cells down) raises or produces garbage text, stop --**
do not install peft/trl or spend GPU time. See the plan's Risks section first.

In [ ]:
HF_USERNAME = 'CHANGE_ME'  # <-- set this before running
HF_REPO_SFT = f'{HF_USERNAME}/socratic-tutor-sft-v6-fp16'      # input: fused SFT checkpoint
HF_REPO_DPO_OUT = f'{HF_USERNAME}/socratic-tutor-dpo-v1-fp16'  # output: merged DPO checkpoint
PAIRS_PATH = 'dpo_pairs_rendered.jsonl'  # upload via the Colab file browser, or a Drive path

In [ ]:
!pip install -q "transformers==4.57.6" "peft>=0.13" "bitsandbytes>=0.44" "trl>=0.12" accelerate datasets huggingface_hub

## Auth

Reads `HF_TOKEN` from Colab's Secrets pane (key icon, left sidebar) -- set it once per
Google account, toggle notebook access on. Needs `write` scope since this notebook pushes
the merged DPO output at the end.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN'))

## Step 1 -- fail-fast load check

This is the highest-risk step in the whole DPO pipeline: confirming the `mlx_lm fuse
--dequantize` output actually loads cleanly in `transformers`. If this cell raises, or the
generated text looks like garbage/repeated tokens, STOP -- do not install peft/trl or spend
GPU time. The most likely failure mode is a `config.json` field gap (`tie_word_embeddings`
plus the correspondingly-dropped `lm_head.weight`), not a tensor-key mismatch -- see the
plan's Risks section for the exact fallback (patch `config.json` against `Qwen/Qwen3-1.7B`'s
canonical one, or as a last resort a manual `load_state_dict(strict=False)` key diff).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

_probe_model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_SFT, torch_dtype='auto', device_map='cuda'
)
tok = AutoTokenizer.from_pretrained(HF_REPO_SFT)

_inputs = tok('The capital of France is', return_tensors='pt').to(_probe_model.device)
_out = _probe_model.generate(**_inputs, max_new_tokens=20)
print(tok.decode(_out[0], skip_special_tokens=True))
# Expect something coherent (e.g. '...Paris.'). Garbage/repeated-token output means the
# fuse output didn't transfer correctly -- stop and check the plan's Risks section.

del _probe_model
import gc, torch
gc.collect()
torch.cuda.empty_cache()

## Step 2 -- reload in 4-bit, attach a fresh LoRA

This LoRA is unrelated to the local MLX adapter format -- a brand-new PEFT adapter for the
DPO pass only. Rank halved vs. the SFT run's r=16: the pair set here is a much smaller data
regime than SFT's training set, and lower rank reduces overfit risk.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# bf16 compute pairs with bf16=True in the trainer (no GradScaler -> no fp16/bf16
# unscale mismatch). bf16 has fp32's exponent range, so it needs no gradient scaling.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_SFT, quantization_config=bnb_config, device_map='cuda'
)

peft_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 3 -- load the rendered pairs, hold out an eval split

`dpo_pairs_rendered.jsonl` already has `prompt`/`chosen`/`rejected` columns rendered exactly
through this project's SFT chat-template pipeline (verified locally: the dense
`Qwen/Qwen3-1.7B` tokenizer's chat template matches `mlx-community/Qwen3-1.7B-4bit`'s
byte-for-byte). No further prompt construction needed here -- these three columns are
directly what `trl.DPOTrainer` expects.

In [ ]:
import json
import random
from datasets import Dataset

with open(PAIRS_PATH) as f:
    rows = [json.loads(line) for line in f]
print(f'loaded {len(rows)} pairs')

random.Random(0).shuffle(rows)
n_eval = max(1, int(len(rows) * 0.15))
eval_rows, train_rows = rows[:n_eval], rows[n_eval:]
print(f'{len(train_rows)} train / {len(eval_rows)} eval')

def to_ds(rs):
    return Dataset.from_list(
        [{'prompt': r['prompt'], 'chosen': r['chosen'], 'rejected': r['rejected']} for r in rs]
    )

train_ds, eval_ds = to_ds(train_rows), to_ds(eval_rows)

## Step 4 -- mount Drive for checkpointing

Not strictly necessary at this size (a couple hundred pairs x 2 epochs is likely well under
15 minutes on a T4), but near-zero cost and survives an unexpected disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/dpo_ckpt', exist_ok=True)

## Step 5 -- DPO training (reference-free)

`ref_model=None` with a PEFT model: current `trl` computes reference-policy logprobs by
disabling the adapter internally rather than loading a second full model copy -- needed to
fit comfortably on a T4. The cell below checks whether the installed `trl` version shows
this behavior; if not, uncomment `precompute_ref_log_probs=True` in `DPOConfig` as a
fallback.

Hyperparameters for this small-data regime:
- `beta=0.1` -- moderate KL penalty (trl's common default); don't go lower, it risks
  overfitting a sparse preference signal.
- `learning_rate=5e-6` -- about 20x below the SFT run's 1e-4; DPO here is a light deepening
  pass on top of SFT, not primary learning.
- `num_train_epochs=2` -- watch the eval-split reward margin (logged by trl) for
  stagnation/collapse as the real stop signal, not train loss.

In [ ]:
import inspect
import dataclasses
import trl
from trl import DPOConfig, DPOTrainer

print('trl version:', trl.__version__)
_src = inspect.getsource(trl.DPOTrainer)  # whole class -- robust to trl method renames
print('adapter-disable reference path detected:',
      'disable_adapter' in _src or 'is_peft_model' in _src)

# trl's DPOConfig field set drifts across versions (e.g. max_prompt_length came/went).
# Filter to whatever this installed version actually accepts so construction never
# TypeErrors on an unexpected kwarg; report anything dropped.
_valid = {f.name for f in dataclasses.fields(DPOConfig)}
_desired = dict(
    beta=0.1,
    learning_rate=5e-6,
    num_train_epochs=2,
    # DPO converts the full [batch*2, seq, vocab] logits to fp32 each forward; with a
    # 151k vocab + ~1024-token seqs that tensor is the memory bottleneck. batch 2 fits
    # a 16GB T4 / 22GB L4; batch 4 OOMs. eval batch defaults to 8 -> set it too, or the
    # epoch-end eval OOMs even when training fits.
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    bf16=True,  # pairs with bf16 compute in the reload cell; no GradScaler (see note there)
    output_dir='/content/drive/MyDrive/dpo_ckpt',
    report_to='none',
    max_length=1024,
    max_prompt_length=768,
    # precompute_ref_log_probs=True,  # fallback if adapter-disable path prints False
)
_kwargs = {k: v for k, v in _desired.items() if k in _valid}
_dropped = sorted(set(_desired) - set(_kwargs))
if _dropped:
    print('NOTE: dropped kwargs not in your trl DPOConfig:', _dropped)
    print('      available length-related fields:',
          sorted(f for f in _valid if 'length' in f or 'len' in f))

dpo_config = DPOConfig(**_kwargs)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tok,
)

trainer.train()

**Resumed after a disconnect?** Re-run the config/installs/auth cells, then the fail-fast
and 4-bit-reload cells (skip re-eyeballing the probe generation if already confirmed once),
then the dataset and Drive-mount cells. In the training cell, replace `trainer.train()` with
`trainer.train(resume_from_checkpoint=True)` -- trl auto-discovers the latest checkpoint
under `output_dir`.

## Step 6 -- merge and push

Folds the DPO LoRA into the base weights and pushes a standalone dense checkpoint -- this
is what `mlx_lm convert` reads back into MLX locally in Phase C.

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Merge the DPO adapter into an FP16-loaded base, NOT the 4-bit training model.
# merge_and_unload() on a 4-bit (bitsandbytes) base re-quantizes back to 4-bit and
# saves bnb quant buffers (weight.absmax / quant_state.bitsandbytes__nf4 / ...),
# which `mlx_lm convert` cannot read. Reloading the base in fp16 first yields a
# clean dense checkpoint.

# 1. Save just the trained LoRA adapter (small), then free the 4-bit training model.
trainer.model.save_pretrained('/content/dpo_adapter')
del trainer, model
gc.collect(); torch.cuda.empty_cache()

# 2. Reload the SFT base in fp16 and merge the adapter into it -> dense fp16.
base = AutoModelForCausalLM.from_pretrained(HF_REPO_SFT, torch_dtype=torch.float16, device_map='cpu')
merged = PeftModel.from_pretrained(base, '/content/dpo_adapter').merge_and_unload()

# 3. Push the dense fp16 checkpoint (this is what mlx_lm convert reads back locally).
merged.save_pretrained('/content/dpo_merged')
tok.save_pretrained('/content/dpo_merged')
merged.push_to_hub(HF_REPO_DPO_OUT, private=True)
tok.push_to_hub(HF_REPO_DPO_OUT)
print(f'pushed dense fp16 to https://huggingface.co/{HF_REPO_DPO_OUT}')

## Next: back to local

On your machine (Phase C -- `scripts/postprocess_dpo_colab.py` wraps this):
```
python -m mlx_lm convert --hf-path <HF_REPO_DPO_OUT> -q --dtype float16 \
    --mlx-path data/dpo/mlx_dpo_v1_4bit
python scripts/eval_harness.py --test eval/gold/test.jsonl \
    --model data/dpo/mlx_dpo_v1_4bit --tag dpo_v1 --out eval/results/dpo_v1
```
Compare `dpo_v1`'s rewrite-safety numbers against the SFT-only `adapters/v6` run on the same
`eval/gold/test.jsonl` -- the whole point of this stage is whether rewrite-safety moves off
its ~1.0 leak-rate plateau.